### ChatBot Evaluation

In [3]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [4]:
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY", "")

In [5]:
## create the datapoints

from langsmith import Client
client = Client()

## define the dataset - test data
dataset_name = "Simple Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)

examples = [
    {
        "inputs": {"question": "What is LangChain?"},
        "outputs": {"answer": "A framework for building LLM applications"},
    },
    {
        "inputs": {"question": "What is LangSmith?"},
        "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
    },
    {
        "inputs": {"question": "What is OpenAI?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    },
    {
        "inputs": {"question": "What is Google?"},
        "outputs": {"answer": "A technology company known for search"},
    },
    {
        "inputs": {"question": "What is Mistral?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    }
]

client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['46af3c81-0b10-4236-b292-3bbb9c674d94',
  '56b31950-29a2-471e-9b05-55600914028b',
  'afca66ca-4a18-4ead-99e0-292f8b285fa1',
  '29afca75-e659-4aaf-a6db-12f045fa608e',
  'cf8cf1e8-37cf-43de-8c74-257f6478a958'],
 'count': 5,
 'as_of': '2026-07-11T20:04:07.758972486Z'}

### Define Metrics (LLM as a Judge)

In [6]:
from google import genai
from langsmith import wrappers

gemini_client = wrappers.wrap_gemini(genai.Client(api_key=os.environ.get("GEMINI_API_KEY")))

C:\Users\agarw\AppData\Local\Temp\ipykernel_7660\2379368207.py:4: LangSmithBetaWarning: Function wrap_gemini is in beta.
  gemini_client = wrappers.wrap_gemini(genai.Client(api_key=os.environ.get("GEMINI_API_KEY")))


In [7]:
# To list and print all available models
for model in gemini_client.models.list():
    print(model.name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

In [8]:
eval_instructions = "You are an expert professor specialized in grading students' answers to questions"

In [9]:
from google.genai import types

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""
        Your are grading the following question:
        {inputs['question']}
        Here is the real answer:
        {reference_outputs['answer']}
        You are grading the following predicted answer:
        {outputs['response']}
        Respond with CORRECT or INCORRECT:
        Grade:
    """

    response = gemini_client.models.generate_content(
        model = "gemini-3.1-flash-lite",
        contents = user_content,
        config = types.GenerateContentConfig(
            system_instruction = eval_instructions,
            temperature = 0.0
        )
    )

    return response.text.strip() == "CORRECT"

In [10]:
## concisions: checks whether the actual output is less than 2x the length of the expected result

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

### Run the Evaluation

In [11]:
default_instructions = "Respond to the users questions in a short, concise manner (one short sentence)"

In [12]:
from google.genai import types

def my_app(question: str, model: str = "gemini-3.1-flash-lite", instructions: str = default_instructions) -> str:
    response = gemini_client.models.generate_content(
        model = model,
        contents = question,
        config = types.GenerateContentConfig(
            system_instruction = instructions,
            temperature = 0.0
        )
    )
    return response.text

In [13]:
### call my_app for every datapoints
def ls_target(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"])}

In [14]:
### test cell
print(ls_target({"question": "Hello"}))

c:\Users\agarw\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


{'response': 'Hello! How can I help you today?'}


In [15]:
### run our evaluation

experiment_results = client.evaluate(
    ls_target,
    data = dataset_name,
    evaluators = [correctness, concision],
    experiment_prefix="gemini-3.1-flash-lite-mini-chatbot"
)

c:\Users\agarw\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'gemini-3.1-flash-lite-mini-chatbot-23a4cde4' at:
https://smith.langchain.com/o/5f471f6c-6457-4e71-a580-9b9051eba12f/datasets/901b9e28-1f2b-4719-bda8-7e1e4cb3e0c3/compare?selectedSessions=9ce5aab8-ba32-40ec-ae85-35949f08cb00




5it [00:06,  1.35s/it]


### Evaluation for RAG

In [16]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap = 0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

# Reduce the number of document splits to process to stay within API quota
# The free tier quota for 'embed_content_free_tier_requests' is 100.
# We'll use a small subset here, e.g., the first 20 splits.
doc_splits_subset = doc_splits[:20]

# Add the document chunks to the "vector store" using GoogleGenerativeAIEmbeddings
vectorStore = InMemoryVectorStore.from_documents(
    documents=doc_splits_subset,
    embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"),
)

# With lanchain we can easily turn any vector store into a retrieval component:
retriever = vectorStore.as_retriever(k=6)

C:\Users\agarw\AppData\Local\Temp\ipykernel_7660\1021585474.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


ModuleNotFoundError: No module named 'langchain_google_genai'

In [17]:
retriever.invoke("what is an agent?")

NameError: name 'retriever' is not defined

In [18]:
from langchain.chat_models import init_chat_model

In [19]:
llm = init_chat_model("gemini-3.1-flash-lite", model_provider="google_genai")
llm

ImportError: Initializing ChatGoogleGenerativeAI requires the langchain-google-genai package. Please install it with `pip install langchain-google-genai`

In [ ]:
from langsmith import traceable

# add decorator
@traceable()
def rag_bot(question: str) -> dict:
    # relevant context
    docs = retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.
    Documents:
    {docs_string}"""

    # llm invoke 

    ai_msg = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ])
    return {"answer": ai_msg.content, "documents ": docs}

In [ ]:
rag_bot("what is an agent?")

### Dataset

In [20]:
from langsmith import Client

client = Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

dataset_name = "RAG Test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id = dataset.id,
    examples = examples
)

{'example_ids': ['543a3e09-37bc-4088-904b-abe4e9ff2e70',
  'b3fdc4b5-745c-4d18-bf6f-82a916c257dc',
  'b4fd88bf-8239-48a7-8497-f62c797743d5'],
 'count': 3,
 'as_of': '2026-07-11T20:06:31.889865962Z'}

### Evaluators or Metrics

In [21]:
from typing_extensions import Annotated, TypedDict

## Correctness Output Schema

# Grade output schema
class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

## correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

In [22]:
from langchain_google_genai import ChatGoogleGenerativeAI

grader_llm = ChatGoogleGenerativeAI(
                model = "gemini-3.1-flash-lite",
                temperature = 0
            ).with_structured_output (
                CorrectnessGrade,
                method="json_schema",
                strict = True 
                )

ModuleNotFoundError: No module named 'langchain_google_genai'

In [ ]:
## evaluator

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for the RAG answer accuracy"""
    answers = f"""\
        QUESTION: {inputs['question']}
        GROUND TRUTH ANSWER: {reference_outputs['answer']}
        STUDENT ANSWER: {outputs['answer']}"""
    
    # run evaluator 
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])
    return grade["correct"]

### Relevance: Response vs Input

In [23]:
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

relevance_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset.
"""

In [24]:
relevance_llm = ChatGoogleGenerativeAI(
                model = "gemini-3.1-flash-lite",
                temperature = 0
            ).with_structured_output (
                RelevanceGrade,
                method="json_schema",
                strict = True 
                )

NameError: name 'ChatGoogleGenerativeAI' is not defined

In [25]:
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"

    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])

    return grade["relevant"]

### Groundedness : Response vs Retrieved docs

In [26]:
class GroundedGrade (TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"] 

grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

In [27]:
grounded_llm = ChatGoogleGenerativeAI(
                model = "gemini-3.1-flash-lite",
                temperature = 0
            ).with_structured_output (
                GroundedGrade,
                method="json_schema",
                strict = True 
                )

NameError: name 'ChatGoogleGenerativeAI' is not defined

In [28]:
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

### Retrieval Relevance: Retrieved docs vs input

In [29]:
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

In [30]:
retrieval_relevance_llm = ChatGoogleGenerativeAI(
        model="gemini-3.1-flash-lite", 
        temperature=0
    ).with_structured_output(
            RetrievalRelevanceGrade, 
            method="json_schema", 
            strict=True
        )

NameError: name 'ChatGoogleGenerativeAI' is not defined

In [31]:
def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### Run the evaluation

In [32]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gemini-3.1-flash-lite"},
)
# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-79397825' at:
https://smith.langchain.com/o/5f471f6c-6457-4e71-a580-9b9051eba12f/datasets/6e30d692-52c6-469a-8fdc-abf70ed8d205/compare?selectedSessions=9002d4ac-b854-478e-9d5b-6f04555c717b




0it [00:00, ?it/s]Error running target function: name 'rag_bot' is not defined
Traceback (most recent call last):
  File "c:\Users\agarw\AppData\Local\Programs\Python\Python314\Lib\site-packages\langsmith\evaluation\_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
    ~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\agarw\AppData\Local\Programs\Python\Python314\Lib\site-packages\langsmith\run_helpers.py", line 777, in wrapper
    function_result = run_container["context"].run(
        func, *args, **kwargs
    )
  File "C:\Users\agarw\AppData\Local\Temp\ipykernel_7660\326724933.py", line 2, in target
    return rag_bot(inputs["question"])
           ^^^^^^^
NameError: name 'rag_bot' is not defined
Error running evaluator <DynamicRunEvaluator correctness> on run 019f52c9-f9c4-7f90-9a9e-28b20f9db7a0: KeyError('response')
Traceback (most recent call last):
  File "c:\Users\agarw\AppData\Local\Programs\Python\Python314\Lib\site-packages\langsmit

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What are the types of biases that can arise wi...,None,"NameError(""name 'rag_bot' is not defined"")\n\n...",The biases that can arise with few-shot prompt...,None,None,None,None,0.024530,543a3e09-37bc-4088-904b-abe4e9ff2e70,019f52c9-f9c4-7f90-9a9e-28b20f9db7a0
1,What are five types of adversarial attacks?,None,"NameError(""name 'rag_bot' is not defined"")\n\n...",Five types of adversarial attacks are (1) Toke...,None,None,None,None,0.001524,b3fdc4b5-745c-4d18-bf6f-82a916c257dc,019f52c9-fa06-7963-9d0d-ef3a3b2be90d
2,How does the ReAct agent use self-reflection?,None,"NameError(""name 'rag_bot' is not defined"")\n\n...","ReAct integrates reasoning and acting, perform...",None,None,None,None,0.001558,b4fd88bf-8239-48a7-8497-f62c797743d5,019f52c9-fa2d-7b60-9454-56386426cf59
